In [4]:
from langchain_classic.storage import LocalFileStore 
from langchain_openai import OpenAIEmbeddings
from langchain_classic.embeddings.cache import CacheBackedEmbeddings
from langchain_community.vectorstores.faiss import FAISS
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("08-Embeddings")

embedding = OpenAIEmbeddings()

store = LocalFileStore("./cache/")

LangSmith 추적을 시작합니다.
[프로젝트명]
08-Embeddings


In [5]:
cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings= embedding,
    document_embedding_cache=store,
    namespace=embedding.model,
)

d:\rag_one\.venv\Lib\site-packages\langchain_classic\embeddings\cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()


In [6]:
list(store.yield_keys())

[]

In [8]:
from langchain_classic.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

raw_documents = TextLoader("../data/appendix-keywords.txt",encoding="utf-8").load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
documents = text_splitter.split_documents(raw_documents)

In [11]:
%time db = FAISS.from_documents(documents, cached_embedder)

CPU times: total: 46.9 ms
Wall time: 644 ms


In [12]:
%time db2 = FAISS.from_documents(documents, cached_embedder)

CPU times: total: 0 ns
Wall time: 7 ms


In [ ]:
# 비영구적인 방법.
from langchain_classic.embeddings.cache import CacheBackedEmbeddings
from langchain_classic.storage import InMemoryByteStore

store = InMemoryByteStore()

cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    embedding, store, namespace=embedding.model
)
